In [ ]:
#License: MIT
# Atom detection adapted from gduscher MLSTEM2025:
# https://github.com/gduscher/MLSTEM2025/blob/main/Day%201/5_Atom_Finding.ipynb

%matplotlib widget

import matplotlib.pylab as plt
import numpy as np

import pyTEMlib
import pyTEMlib.file_tools
import pyTEMlib.image_tools
import pyTEMlib.probe_tools
import pyTEMlib.atom_tools

import skimage.feature

In [ ]:
# load the calibrated aligned image written by single_frame.ipynb
folder = '/path/to/folder/'
datasets = pyTEMlib.file_tools.open_file(folder + 'aligned.hf5')
aligned = datasets[list(datasets)[0]]   # averaged registered frame; carries .x.slope


In [ ]:
# ------- Input ------
#atoms_size = .065 # in nm
atoms_size = .075 # in nm
# --------------------

image = aligned

out_tags = {}
#image.metadata['experiment']= {'convergence_angle': 30, 'acceleration_voltage': 200000.}

In [ ]:
#https://github.com/gduscher/MLSTEM2025/blob/main/Day%201/5_Atom_Finding.ipynb



scale_x = image.x.slope
gauss_diameter = atoms_size/scale_x
gauss_probe = pyTEMlib.probe_tools.make_gauss(image.shape[0], image.shape[1], gauss_diameter)

print('Deconvolution of ', aligned.title)
LR_dataset = pyTEMlib.image_tools.decon_lr(image, gauss_probe, verbose=False)
datasets['Log_001'] = LR_dataset
extent = LR_dataset.get_extent([0,1])
fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(image.T, extent = extent,vmax=np.median(np.array(image))+3*np.std(np.array(image)))
ax[1].imshow(LR_dataset.T, extent = extent, vmax=np.median(np.array(LR_dataset))+3*np.std(np.array(LR_dataset)));


In [ ]:

# ------- Input ------
threshold = 0.9 #usally between 0.01 and 0.9  the smaller the more atoms
atom_size = .06 #in nm  
# ----------------------


image = LR_dataset
#image = image_choice.dataset
scale_x = image.x.slope
blobs =  skimage.feature.blob_log(image, max_sigma=atom_size/scale_x, threshold=threshold)

fig1, ax = plt.subplots(1, 1,figsize=(8,7), sharex=True, sharey=True)
plt.title("blob detection ")

plt.imshow(image.T, interpolation='nearest',cmap='gray', vmax=np.median(np.array(image))+3*np.std(np.array(image)))
plt.scatter(blobs[:, 0], blobs[:, 1], c='r', s=20, alpha = .5);


In [ ]:
atoms = blobs
image = aligned
image = image-image.min()

atom_radius = 2
MaxInt = 0
MinInt = 0
maxDist = 2
sym = pyTEMlib.atom_tools.atom_refine(np.array(image), atoms, atom_radius, max_int = 0, min_int = 0, max_dist = 2)
refined_atoms = np.array(sym['atoms'])

fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
ax[0].imshow(image.T)
ax[0].scatter(refined_atoms[:,0],refined_atoms[:,1],  s=10, alpha = 0.3, color = 'red')
ax[0].set_title('refined atom postion')
ax[1].imshow(image.T)
ax[1].scatter(atoms[:, 0], atoms[:, 1], c='r', s=10, alpha = .3);
ax[1].set_title('blobs on image');


In [ ]:
plt.close('all')
plt.violinplot(sym['gauss_intensity'])
plt.show()

In [ ]:
# Hand off to vmap's detect_columns for the atom-column refinement.
# Seeds (deconv -> blob_log -> atom_refine) are written as an x_obs0/y_obs0 CSV;
# the aligned image is saved as a hyperspy-readable TIFF, transposed (image.T)
# so detect_columns' Sublattice(seed, image=s) reproduces the atomap (x,y)
# convention this notebook used previously.
import pandas as pd
import hyperspy.api as hs

img = np.ascontiguousarray(np.asarray(image).T)
slope = aligned.x.slope                 # nm/px (square-pixel assumption, as in vmap)
imsize = [img.shape[0] * slope, img.shape[1] * slope]

hs.signals.Signal2D(img).save(folder + 'aligned.tif', overwrite=True)
pd.DataFrame({'x_obs0': refined_atoms[:, 0],
              'y_obs0': refined_atoms[:, 1]}).to_csv(folder + 'seed.csv', index=False)

In [ ]:
# vmap-native column refinement (CoM + 2D-gaussian refine + column integration).
# Writes folder + 'aligned_xyI.csv' (x_obs0,y_obs0,ell0,rot0,I0,...) -- the standard
# vmap interchange CSV consumed by the lattice-refinement pipeline (and start_csv).
import sys
sys.path.append('../vector_maps')       # sibling dir in STEM_tools; adjust if needed
from detect_columns import detect_columns

detect_columns(
    fname='aligned.tif',
    folder=folder,
    imsize=imsize,
    sep=8,                              # cosmetic when start_csv seeds the detection
    start_csv=folder + 'seed.csv',
    ptonn=0.4,                          # = the notebook's previous percent_to_nn
)

In [ ]:
fft_image = aligned.fft().abs()
fft_image.plot()

In [ ]:
aligned.shape